## 파일불러오기(cluster 수정해야함 지금 0,1값만 있어)

In [ ]:
import geopandas as gpd

# 지정된 경로에서 gdf_result 불러오기
file_path = r"gdf_result.gpkg"
gdf_result = gpd.read_file(file_path)

# 데이터 확인
gdf_result.head()


In [ ]:
import pandas as pd

In [ ]:
df2 = pd.read_csv('preprocessed_busan_data_visualization.csv')

In [ ]:
gdf_result['cluster'] = df2['cluster']

In [ ]:
 gdf_result.columns

In [ ]:
gdf_result.info()

## 변수7개(일단 토지혼합도 포함해서 했음) 지도 시각화

- 행정동별 시각화라 qgis 필요없고 matplotlib이 더 직관적인 것 같아서 여기다 함 
- 위는 변수명 / 아래는 보고서상 최종변수

## count_per_km 에서 보듯이 극단값으로 인해서 시각화가 잘 안되는 것들이 몇 있음 scaling이나 log 고려해야할듯?!
- 하고 다시해야할 것 같아서 코드 넘깁니다

In [ ]:
pip install folium

In [ ]:
pip install matplotlib

In [ ]:
import folium
import matplotlib.cm as cm
import matplotlib.colors as colors
from folium.features import GeoJson, GeoJsonTooltip

# 각 피처별 컬러맵 지정
features_and_cmaps = {
    'floating_resident_ratio': 'YlGnBu',
    'youth_floating_ratio': 'OrRd',
    'floating_pop_17_20': 'PuBuGn',
    'searching_average': 'BuPu',
    'HubDist': 'YlOrBr',
    'count_per_km': 'Greens',
    'entropy_index': 'Greys'
}

# 각 피처에 대해 독립적으로 지도 생성 및 저장
for feature, cmap_name in features_and_cmaps.items():
    m = folium.Map(location=[35.1796, 129.0756], zoom_start=11)

    # 정규화 및 컬러맵 설정
    cmap = cm.get_cmap(cmap_name)
    min_val = gdf_result[feature].min()
    max_val = gdf_result[feature].max()
    norm = colors.Normalize(vmin=min_val, vmax=max_val)

    # 스타일 함수 정의
    def make_style(value):
        if value is None:
            return '#8c8c8c'
        return colors.to_hex(cmap(norm(value)))

    def style_function(feature_data):
        val = feature_data['properties'].get(feature, None)
        return {
            'fillColor': make_style(val),
            'color': 'black',
            'weight': 1,
            'fillOpacity': 1
        }

    # GeoJson 생성
    gjson = GeoJson(
        gdf_result,
        name=feature,
        style_function=style_function,
        tooltip=GeoJsonTooltip(
            fields=['행정동명', feature],
            aliases=['행정동명:', f'{feature}:'],
            sticky=True
        )
    )
    gjson.add_to(m)

    # 저장
    file_name = f"map_{feature}.html"
    m.save(file_name)
    print(f"✅ 저장됨: {file_name}")

In [ ]:
import folium
import matplotlib.cm as cm
import matplotlib.colors as colors
from folium.features import GeoJson, GeoJsonTooltip

# 피처별 컬러맵 지정
features_and_cmaps = {
    'floating_resident_ratio': 'YlGnBu',
    'youth_floating_ratio': 'OrRd',
    'floating_pop_17_20': 'PuBuGn',
    'searching_average': 'BuPu',
    'HubDist': 'YlOrBr',
    'count_per_km': 'Greens',
    'entropy_index': 'Greys'
}

# 클러스터 ID 목록
cluster_ids = sorted(gdf_result['cluster'].unique())  # 예: [0, 1, 2, 3]

# 각 클러스터에 대해 반복
for cluster_id in cluster_ids:
    gdf_cluster = gdf_result[gdf_result['cluster'] == cluster_id]

    for feature, cmap_name in features_and_cmaps.items():
        m = folium.Map(location=[35.1796, 129.0756], zoom_start=11)

        # 정규화 및 컬러맵 설정
        cmap = cm.get_cmap(cmap_name)
        min_val = gdf_cluster[feature].min()
        max_val = gdf_cluster[feature].max()
        norm = colors.Normalize(vmin=min_val, vmax=max_val)

        # 색상 매핑 함수
        def make_style(value):
            if value is None:
                return '#8c8c8c'
            return colors.to_hex(cmap(norm(value)))

        def style_function(feature_data):
            val = feature_data['properties'].get(feature, None)
            return {
                'fillColor': make_style(val),
                'color': 'black',  # 경계선 색
                'weight': 1,
                'fillOpacity': 1   # 완전 불투명
            }

        # GeoJson 시각화
        gjson = GeoJson(
            gdf_cluster,
            name=f"cluster_{cluster_id}_{feature}",
            style_function=style_function,
            tooltip=GeoJsonTooltip(
                fields=['행정동명', feature],
                aliases=['행정동명:', f'{feature}:'],
                sticky=True
            )
        )
        gjson.add_to(m)

        # 저장
        file_name = f"map_cluster_{cluster_id}_{feature}.html"
        m.save(file_name)
        print(f"✅ 저장됨: {file_name}")

In [ ]:
pip install streamlit streamlit-folium geopandas matplotlib

In [ ]:
import geopandas as gpd
import folium
from folium import FeatureGroup, GeoJson, GeoJsonTooltip
from shapely.geometry import mapping

In [ ]:
# 🔹 클러스터 색상
cluster_colors = {
    0: '#000000',
    1: '#1f77b4',
    2: '#2ca02c',
    3: '#ff7f0e',
}
cluster_ids = sorted(gdf_result['cluster'].unique())

# 🔹 feature 목록
features = [
    'floating_resident_ratio', 'youth_floating_ratio', 'floating_pop_17_20',
    'searching_average', 'HubDist', 'count_per_km', 'entropy_index'
]

# 🔹 Folium 지도 생성
m = folium.Map(location=[35.1796, 129.0756], zoom_start=11)

# ✅ 1. 클러스터별 FeatureGroup 추가
for cluster_id in cluster_ids:
    fg = FeatureGroup(name=f"클러스터 {cluster_id}", show=False)
    cluster_color = cluster_colors.get(cluster_id, "#888888")

    for _, row in gdf_result.iterrows():
        is_target = row['cluster'] == cluster_id
        fc = cluster_color if is_target else 'transparent'
        op = 0.9 if is_target else 0

        def make_style(fill=fc, opacity=op):
            return lambda x: {
                'fillColor': fill,
                'color': 'white',
                'weight': 1,
                'fillOpacity': opacity
            }

        feature = {
            "type": "Feature",
            "geometry": mapping(row['geometry']),
            "properties": {
                "행정동명": row['행정동명'],
                **{f: row[f] for f in features}
            }
        }

        gjson = GeoJson(
            feature,
            style_function=make_style(),
            tooltip=GeoJsonTooltip(
                fields=["행정동명"] + features,
                aliases=["행정동명:"] + [f"{f}:" for f in features],
                sticky=True
            )
        )
        fg.add_child(gjson)

    fg.add_to(m)

# ✅ 2. 피처별 FeatureGroup 추가 (검정색 + fillOpacity)
for feature in features:
    fg = FeatureGroup(name=f"{feature} (검정+높낮이)", show=False)
    min_val = gdf_result[feature].min()
    max_val = gdf_result[feature].max()
    val_range = max_val - min_val if max_val > min_val else 1e-6

    for _, row in gdf_result.iterrows():
        value = row[feature]
        norm_opacity = (value - min_val) / val_range
        norm_opacity = min(max(norm_opacity, 0.2), 1.0)

        def make_opacity_style(opacity=norm_opacity):
            return lambda x: {
                'fillColor': '#000000',
                'color': 'white',
                'weight': 1,
                'fillOpacity': opacity
            }

        feature_dict = {
            "type": "Feature",
            "geometry": mapping(row['geometry']),
            "properties": {
                "행정동명": row['행정동명'],
                **{f: row[f] for f in features}
            }
        }

        gjson = GeoJson(
            feature_dict,
            style_function=make_opacity_style(),
            tooltip=GeoJsonTooltip(
                fields=["행정동명"] + features,
                aliases=["행정동명:"] + [f"{f}:" for f in features],
                sticky=True
            )
        )
        fg.add_child(gjson)

    fg.add_to(m)

# ✅ 레이어 전환 컨트롤 추가
folium.LayerControl(collapsed=False).add_to(m)

# ✅ HTML로 저장
m.save("final_interactive_map.html")
print("✅ 최종 HTML 저장 완료: final_interactive_map.html")

In [ ]:
import folium
from folium import FeatureGroup, GeoJson, GeoJsonTooltip
from shapely.geometry import mapping
import pandas as pd # 예시를 위해 import
import geopandas as gpd # 예시를 위해 import

# --- 이 부분은 사용자의 데이터에 맞게 준비해주세요 (예시 데이터) ---
# gdf_result = gpd.read_file(...)
# gdf_result = gdf_result.to_crs("EPSG:4326") # ★ 좌표계가 다르면 꼭 변환해주세요!
# ----------------------------------------------------------------

# 🔹 클러스터 색상
cluster_colors = {
    0: '#000000',
    1: '#1f77b4',
    2: '#2ca02c',
    3: '#ff7f0e',
}
# 'cluster' 컬럼이 gdf_result에 있다고 가정합니다.
cluster_ids = sorted(gdf_result['cluster'].unique())

# 🔹 feature 목록
features = [
    'floating_resident_ratio', 'youth_floating_ratio', 'floating_pop_17_20',
    'searching_average', 'HubDist', 'count_per_km',
]

# 🔹 Folium 지도 생성
m = folium.Map(location=[35.1796, 129.0756], zoom_start=11)

# ✅ 1. 클러스터별 FeatureGroup 추가 (수정된 방식)
for cluster_id in cluster_ids:
    fg = FeatureGroup(name=f"클러스터 {cluster_id}", show=False)
    color = cluster_colors.get(cluster_id, "#888888")
    
    # 하나의 스타일 함수가 feature의 속성을 보고 스타일을 결정합니다.
    def style_function_cluster(feature, cl_id=cluster_id, cl_color=color):
        if feature['properties']['cluster'] == cl_id:
            return {'fillColor': cl_color, 'color': 'white', 'weight': 1, 'fillOpacity': 0.8}
        else:
            # 해당 클러스터가 아니면 아예 그리지 않음
            return {'fillOpacity': 0, 'weight': 0, 'color': 'transparent'}

    GeoJson(
        gdf_result,
        style_function=style_function_cluster,
        tooltip=GeoJsonTooltip(
            fields=["행정동명", "cluster"] + features,
            aliases=["행정동명:", "클러스터:"] + [f"{f}:" for f in features],
            sticky=True
        )
    ).add_to(fg)
    fg.add_to(m)

# ✅ 2. 피처별 FeatureGroup 추가 (수정된 방식)
for feature_name in features:
    fg = FeatureGroup(name=f"{feature_name} (검정+높낮이)", show=False)
    min_val = gdf_result[feature_name].min()
    max_val = gdf_result[feature_name].max()
    val_range = max_val - min_val if max_val > min_val else 1e-6

    # 클로저를 이용해 feature별 정보를 스타일 함수에 안전하게 전달합니다.
    def make_style_function_feature(fname, v_min, v_range):
        def style_function(feature):
            value = feature['properties'][fname]
            opacity = (value - v_min) / v_range if v_range > 0 else 0.2
            opacity = min(max(opacity, 0.2), 1.0) # 투명도를 0.2 ~ 1.0 사이로 조정
            return {'fillColor': '#000000', 'color': 'white', 'weight': 1, 'fillOpacity': opacity}
        return style_function

    GeoJson(
        gdf_result,
        style_function=make_style_function_feature(feature_name, min_val, val_range),
        tooltip=GeoJsonTooltip(
            fields=["행정동명"] + features,
            aliases=["행정동명:"] + [f"{f}:" for f in features],
            sticky=True
        )
    ).add_to(fg)
    fg.add_to(m)

# ✅ 레이어 전환 컨트롤 추가
folium.LayerControl(collapsed=False).add_to(m)

# ✅ HTML로 저장
m.save("final_interactive_map_revised.html")
print("✅ 최종 HTML 저장 완료: final_interactive_map_revised.html")

In [ ]:
import folium
from folium import FeatureGroup, GeoJson, GeoJsonTooltip, GeoJsonPopup
import json
import pandas as pd
import geopandas as gpd

# --- 사용자 데이터 준비 ---
# gdf_result = gpd.read_file(...)
# gdf_result = gdf_result.to_crs("EPSG:4326")
# -------------------------

# 🔹 클러스터 색상 및 피처 목록
cluster_colors = {0: '#000000', 1: '#1f77b4', 2: '#2ca02c', 3: '#ff7f0e'}
cluster_ids = sorted(gdf_result['cluster'].unique())
features = [
    'floating_resident_ratio', 'youth_floating_ratio', 'floating_pop_17_20',
    'searching_average', 'HubDist', 'count_per_km',
]

# 🔹 Folium 지도 생성
m = folium.Map(location=[35.1796, 129.0756], zoom_start=11, control_scale=True)

# ✅ 1. 클러스터별 보기 (툴팁, 팝업 기능)
all_fields = ["행정동명", "cluster"] + features
all_aliases = ["행정동명:", "클러스터:"] + [f"{f}:" for f in features]

for cluster_id in cluster_ids:
    fg = FeatureGroup(name=f"클러스터 {cluster_id}", show=False)
    
    gjson = GeoJson(
        gdf_result,
        style_function=lambda feature, cl_id=cluster_id, cl_color=cluster_colors.get(cluster_id, "#888888"): {
            'fillColor': cl_color, 'color': 'white', 'weight': 1, 'fillOpacity': 0.8
        } if feature['properties']['cluster'] == cl_id else {
            'fillOpacity': 0, 'weight': 0, 'color': 'transparent', 'interactive': False
        },
        tooltip=GeoJsonTooltip(fields=all_fields, aliases=all_aliases, sticky=True),
        popup=GeoJsonPopup(fields=all_fields, aliases=all_aliases)
    )
    gjson.add_to(fg)
    fg.add_to(m)

# ✅ 2. 피처별 시각화 (라디오 버튼 기능)
feature_layer_vars = {}
is_first_feature = True
for feature_name in features:
    min_val, max_val = gdf_result[feature_name].min(), gdf_result[feature_name].max()
    val_range = max_val - min_val if max_val > min_val else 1

    gjson_layer = folium.GeoJson(
        gdf_result,
        name=f"{feature_name} 기준",
        style_function=lambda feature, fname=feature_name, v_min=min_val, v_range=val_range: {
            'fillColor': '#000000', 'color': 'white', 'weight': 1,
            'fillOpacity': min(max(((feature['properties'][fname] - v_min) / v_range), 0.2), 1.0)
        },
        show=is_first_feature
    ).add_to(m)
    
    feature_layer_vars[feature_name] = gjson_layer.get_name()
    is_first_feature = False

# ✅ 3. 주요 지점(POI) 레이어
poi_data = {
    'name': ['서면역', '부산역', '해운대역', '센텀시티역'],
    'lat': [35.1578, 35.1149, 35.1629, 35.1687],
    'lon': [129.0593, 129.0423, 129.1604, 129.1292]
}
df_poi = pd.DataFrame(poi_data)
poi_group = FeatureGroup(name="주요 지하철역", show=True)
for i, row in df_poi.iterrows():
    folium.Marker(
        location=[row['lat'], row['lon']],
        popup=row['name'],
        icon=folium.Icon(icon='subway', prefix='fa', color='blue')
    ).add_to(poi_group)
poi_group.add_to(m)

# ✅ 4. 범례
legend_html = """
<div style="position: fixed;
     bottom: 50px; right: 50px; width: 150px; height: 90px;
     border:2px solid grey; z-index:9999; font-size:14px;
     background-color:white;
     ">&nbsp; <b>피처별 보기 범례</b><br>
     &nbsp; 값이 낮음 &nbsp; <i class="fa fa-square" style="color:#D3D3D3"></i><br>
     &nbsp;  &nbsp; &nbsp; &nbsp; &nbsp; &nbsp; &nbsp; &#8595; <br>
     &nbsp; 값이 높음 &nbsp; <i class="fa fa-square" style="color:#000000"></i>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))


# ✅ 5. 동적 툴팁을 위한 JavaScript 코드
js_features = json.dumps(features)
js_layer_vars = json.dumps(feature_layer_vars)

script = f"""
<script>
    document.addEventListener("DOMContentLoaded", function() {{
        const layerVars = {js_layer_vars};
        const allFeatures = {js_features};
        const map = {m.get_name()};

        let selectedFeature = allFeatures[0];

        function createTooltipHtml(properties) {{
            let html = `<b>${{properties['행정동명']}}</b><hr style="margin: 3px 0; border-top: 1px solid grey;">`;
            allFeatures.forEach(fname => {{
                const val = properties[fname];
                const displayVal = (typeof val === 'number') ? val.toFixed(2) : val;
                if (fname === selectedFeature) {{
                    html += `<b>${{fname}}: ${{displayVal}}</b><br>`;
                }} else {{
                    html += `${{fname}}: ${{displayVal}}<br>`;
                }}
            }});
            return html;
        }}

        function bindDynamicTooltip(layerVarName) {{
            const layer = eval(layerVarName); 
            if (layer) {{
                layer.eachLayer(function(sublayer) {{
                    sublayer.bindTooltip(function(l) {{
                        return createTooltipHtml(l.feature.properties);
                    }}, {{sticky: true}});
                }});
            }}
        }}
        
        Object.values(layerVars).forEach(bindDynamicTooltip);

        map.on('overlayadd', function(e) {{
            if (e.name.includes(' 기준')) {{
                selectedFeature = e.name.replace(' 기준', '');
            }}
        }});
    }});
</script>
"""
m.get_root().html.add_child(folium.Element(script))

# 최종 레이어 컨트롤 추가 및 저장
folium.LayerControl(collapsed=False).add_to(m)
m.save("final_map_fully_fixed.html")
print("✅ 모든 오류가 수정된 최종 HTML 저장 완료: final_map_fully_fixed.html")

In [ ]:
import folium
import matplotlib.colors as colors
from shapely.geometry import mapping
from folium import FeatureGroup, GeoJson, GeoJsonTooltip

# 클러스터 색상
cluster_colors = {
    0: '#000000',
    1: '#1f77b4',
    2: '#2ca02c',
    3: '#ff7f0e',
}

# 피처 목록
feature_cols = [
    'floating_resident_ratio', 'youth_floating_ratio', 'floating_pop_17_20',
    'searching_average', 'HubDist', 'count_per_km', 'entropy_index'
]

# 지도 생성
m = folium.Map(location=[35.1796, 129.0756], zoom_start=11)

# ✅ 1. 클러스터 단색 모드
for cluster_id, color in cluster_colors.items():
    fg = FeatureGroup(name=f"클러스터 {cluster_id} (단색)", show=False)
    gdf_cluster = gdf_result[gdf_result['cluster'] == cluster_id]

    for _, row in gdf_cluster.iterrows():
        geojson_feature = {
            "type": "Feature",
            "properties": {
                "행정동명": row['행정동명'],
                **{f: row[f] for f in feature_cols}
            },
            "geometry": mapping(row['geometry'])
        }

        # 🔧 color 캡처
        def make_style_func(c=color):
            return lambda x: {
                'fillColor': c,
                'color': 'black',
                'weight': 1,
                'fillOpacity': 0.8
            }

        GeoJson(
            geojson_feature,
            style_function=make_style_func(),
            tooltip=GeoJsonTooltip(
                fields=["행정동명"] + feature_cols,
                aliases=["행정동명:"] + [f"{f}:" for f in feature_cols],
                sticky=True
            )
        ).add_to(fg)

    fg.add_to(m)

# ✅ 2. 피처 높낮이 모드 (검정색 + fillOpacity)
for feature in feature_cols:
    fg = FeatureGroup(name=f"{feature} (검정/높낮이)", show=False)

    min_val = gdf_result[feature].min()
    max_val = gdf_result[feature].max()
    value_range = max_val - min_val if max_val > min_val else 1e-6

    for _, row in gdf_result.iterrows():
        value = row[feature]
        norm_opacity = (value - min_val) / value_range  # 0 ~ 1 사이

        geojson_feature = {
            "type": "Feature",
            "properties": {
                "행정동명": row['행정동명'],
                **{f: row[f] for f in feature_cols}
            },
            "geometry": mapping(row['geometry'])
        }

        # 🔧 opacity 캡처
        def make_opacity_func(opacity=norm_opacity):
            return lambda x: {
                'fillColor': '#000000',
                'color': 'black',
                'weight': 1,
                'fillOpacity': opacity
            }

        GeoJson(
            geojson_feature,
            style_function=make_opacity_func(),
            tooltip=GeoJsonTooltip(
                fields=["행정동명"] + feature_cols,
                aliases=["행정동명:"] + [f"{f}:" for f in feature_cols],
                sticky=True
            )
        ).add_to(fg)

    fg.add_to(m)

# ✅ Layer Control
folium.LayerControl(collapsed=False).add_to(m)

# ✅ 저장
m.save("cluster_vs_feature_map_fixed.html")
print("✅ 저장 완료: cluster_vs_feature_map_fixed.html")

In [ ]:
import folium
import matplotlib.cm as cm
import matplotlib.colors as colors
from shapely.geometry import mapping
from folium import FeatureGroup
import geopandas as gpd

# 예시: gdf_result는 GeoDataFrame이어야 합니다
# gdf_result = gpd.read_file('your_geojson_file.geojson')

# 시각화할 feature와 각각의 컬러맵 정의
features_and_cmaps = {
    'floating_resident_ratio': 'YlGnBu',
    'youth_floating_ratio': 'OrRd',
    'HubDist': 'YlOrBr',
}

# folium 지도 생성 (부산 중심 좌표)
m = folium.Map(location=[35.1796, 129.0756], zoom_start=11)

# 각 feature에 대해 FeatureGroup을 생성하고 지도에 추가
for feature, cmap_name in features_and_cmaps.items():
    fg = FeatureGroup(name=feature)
    
    # 컬러맵 설정
    cmap = cm.get_cmap(cmap_name)
    min_val = gdf_result[feature].min()
    max_val = gdf_result[feature].max()
    norm = colors.Normalize(vmin=min_val, vmax=max_val)

    # 각 행마다 GeoJson 생성
    for _, row in gdf_result.iterrows():
        color = colors.rgb2hex(cmap(norm(row[feature])))

        geojson_feature = {
            "type": "Feature",
            "properties": {feature: row[feature]},
            "geometry": mapping(row['geometry'])  # shapely → GeoJSON
        }

        folium.GeoJson(
            geojson_feature,
            style_function=lambda x, color=color: {
                'fillColor': color,
                'color': 'black',
                'weight': 0.5,
                'fillOpacity': 0.6,
            },
            tooltip=folium.GeoJsonTooltip(
                fields=[feature],
                aliases=[feature],
                localize=True
            )
        ).add_to(fg)

    fg.add_to(m)

# 레이어 전환 버튼 추가
folium.LayerControl(collapsed=False).add_to(m)

# 저장 또는 Jupyter에서 바로 보기
m.save('interactive_map.html')
# 또는 Jupyter Notebook이라면 아래로 바로 출력 가능:
# m

In [ ]:
import matplotlib.pyplot as plt

cmap_list = ['YlGnBu', 'OrRd', 'PuBuGn', 'BuPu', 'YlOrBr', 'Greens', 'Greys']
numeric_cols = [
    'floating_resident_ratio', 'youth_floating_ratio', 'floating_pop_17_20',
    'searching_average', 'HubDist', 'count_per_km', 'entropy_index'
]

for i, col in enumerate(numeric_cols):
    ax = gdf_result.plot(
        column=col,
        cmap=cmap_list[i],
        legend=True,
        figsize=(6, 6),  # ✅ 작고 정사각형 비율로
        edgecolor='white'
    )
    plt.title(f'{col}', fontsize=12)
    plt.axis('off')
    plt.tight_layout()

    plt.show()


In [ ]:
import matplotlib.pyplot as plt

# ✅ 변수명 매핑: 영어 변수명 → 한글 설명
col_name_map = {
    'floating_resident_ratio': '유동주거비율',
    'youth_floating_ratio': '청년유동인구비율',
    'floating_pop_17_20': '17–20시 유동인구',
    'searching_average': '검색량 평균',
    'HubDist': '허브 거리',
    'count_per_km': '상점 밀도',
    'entropy_index': '토지 혼합도'
}

# ✅ 시각화에 사용할 변수명과 색상
cmap_list = ['YlGnBu', 'OrRd', 'PuBuGn', 'BuPu', 'YlOrBr', 'Greens', 'gist_yarg']
numeric_cols = list(col_name_map.keys())

# ✅ 한글 폰트 설정 (Windows 기준)
plt.rcParams['font.family'] = 'Malgun Gothic'  # Mac은 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

# ✅ 각 변수별 지도 출력
for i, col in enumerate(numeric_cols):
    ax = gdf_result.plot(
        column=col,
        cmap=cmap_list[i],
        legend=True,
        figsize=(6, 6),
        edgecolor='white'
    )
    plt.title(f'{col_name_map[col]}', fontsize=12)  # ✅ 한글 제목 표시
    plt.axis('off')
    plt.tight_layout()
    plt.show()


## Cluster 시각화(지금 0,1 값밖에 없슈.. 다시 채워넣으면 될듯!)

- 아래는 놀랍게도 나의 reference (서울시 대규모⋅중소규모 점포 매출액 및 위치데이터 기반 머신러닝을 활용한 상권클러스터링)


*(참고 문헌 「서울시 대규모⋅중소규모 점포 매출액 및 위치데이터 기반 머신러닝을 활용한 상권클러스터링」에서 캡처한 이미지는 다른 저자의 저작물이라 공개 저장소에서 삭제했습니다. 원문을 참고하세요.)*

- 위는 2,2 형식 아래는 그림 1개씩

In [ ]:
import matplotlib.pyplot as plt

# 클러스터 색상 (RGBA, 반투명)
cluster_colors = [
    (240/255, 59/255, 32/255, 0.5),     # 🔴 붉은 계열
    (43/255, 140/255, 190/255, 0.5),    # 🔵 파랑 계열
    (161/255, 217/255, 155/255, 0.5),   # 🟢 연녹
    (254/255, 178/255, 76/255, 0.5)     # 🟠 오렌지
]

# 2행 2열 subplot으로 출력
fig, axes = plt.subplots(2, 2, figsize=(12, 12))

for i, ax in enumerate(axes.flat):
    gdf_result.boundary.plot(ax=ax, color='lightgray', linewidth=0.5)
    gdf_result[gdf_result['cluster'] == i].plot(
        ax=ax,
        color=cluster_colors[i],
        edgecolor='black'
    )
    ax.set_title(f'Cluster {i}', fontsize=12)
    ax.axis('off')

plt.tight_layout()
plt.show()


In [ ]:
cluster_colors = [
    (240/255, 59/255, 32/255, 0.5),     # 🔴 붉은 계열 (#f03b20) → 반투명
    (43/255, 140/255, 190/255, 0.5),    # 🔵 파랑 계열 (#2b8cbe)
    (161/255, 217/255, 155/255, 0.5),   # 🟢 연녹 (#a1d99b)
    (254/255, 178/255, 76/255, 0.5)     # 🟠 오렌지 (#feb24c)
]

for i in range(4):
    fig, ax = plt.subplots(figsize=(6, 6))
    
    gdf_result.boundary.plot(ax=ax, color='lightgray', linewidth=0.5)

    gdf_result[gdf_result['cluster'] == i].plot(
        ax=ax,
        color=cluster_colors[i],  # ✅ RGBA 색상 적용
        edgecolor='black'
    )

    plt.title(f'Cluster {i}', fontsize=12)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

## 참고하슈~ (코로나19가 서울시 상권 매출 변화에 미치는 영향 분석 )

*(참고 문헌 「코로나19가 서울시 상권 매출 변화에 미치는 영향 분석」에서 캡처한 이미지는 다른 저자의 저작물이라 공개 저장소에서 삭제했습니다. 원문을 참고하세요.)*

- 표 7 /  표 6은 안했음..

In [ ]:
# 클러스터 명칭 정의 (원하는 명칭으로 바꿔도 됨)
cluster_names = {
    0: '중심업무 상권',
    1: '변화가 상권',
    2: '주거중심 상권',
    3: '임시로 생성'
}

# 요약표 생성
df_num2 = (
    gdf_result.groupby('cluster')['행정동명']
    .count()
    .reset_index()
    .rename(columns={'cluster': '구분', '행정동명': '행정동 수'})
)

# 클러스터 번호를 한글명으로 변환
df_num2['클러스터 명칭'] = df_num2['구분'].map(cluster_names)
df_num2['구분'] = '클러스터' + (df_num2['구분'] + 1).astype(str)
df_num2 = df_num2[['구분', '클러스터 명칭', '행정동 수']]
df_num2